
### Transform Bronze to Silver 
* data type conversion
* data quality validation
* data standardisation
* light feature engineering
* merge open f1 and kaggle datasets 


1. create functions to clean/standardise data first
2. drop duplicates 


In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import *
import re


In [0]:
#change cols to snake case
def rename_cols(df):
    for name in df.columns:
        new_name=(
            re.sub(r'(?<!^)(?=[A-Z])', '_', name)# to change from camelcase
            .replace(" ","_")
            .replace("-","_")
            .lower())
        df=df.withColumnRenamed(name,new_name)
    return df






In [0]:
#view and inspect all tables in bronze

tables=spark.catalog.listTables("f1_warehouse.bronze")

temp_silver={}

for table in tables:
    print('='*80)
    print(table.name)
    
    df=spark.table(f"f1_warehouse.bronze.{table.name}")
    df=rename_cols(df)
    temp_silver[table.name]=df # keep each df in memory as dict and apply final silver overwrite at the end


    print(f"number of rows:{df.count()}")
    print(f"number of columns:{len(df.columns)}")
    duplicate_count=df.count()-df.dropDuplicates().count()
    print(f"number of duplicates:{duplicate_count}")
    df.printSchema() #to check data type and col names

   




observations:
* kaggle uses id and openf1 api uses key
* change column names with key to id
* check if meeting key the same as race id , session id refers to individaul sesssions during one race weekend. check with 2024/2023 data since openf1_sessions and  kaggle has an overlap of 2023/2024 data 
* filter open f1 to be >2025 only
*  explain why time is kept in string



In [0]:
#meeting key the same as race id , session id refers to individaul sesssions during one race weekend. check with 2024/2023 data since openf1_sessions and  kaggle has an overlap of 2023/2024 data 

kaggle_races=(temp_silver["kaggle_races"]
              .filter("year IN (2023,2024)")
              .select("race_id","year","round","name","date")
              .orderBy("date"))

openf1_meetings=(temp_silver["openf1_meetings"]
                 .filter("year IN (2023,2024)")
                 .select("meeting_key","year","meeting_name","date_end")
                 .orderBy("date_start"))




In [0]:
kaggle_races.show(5)

In [0]:
openf1_meetings.show(5)

### observations:


In [0]:
#openf1 data only for >= 2025 except tables (session_results,stints and drivers (dimension table))
#some tables dont have year so use meeting keys where year>=2025

def filter_to_2025(df, meetings_df):
    meetings_2025 = meetings_df.filter("year >= 2025").select("meeting_key")
    return df.join(meetings_2025, on="meeting_key", how="inner")



In [0]:
temp_silver["openf1_meetings"]=(temp_silver["openf1_meetings"].filter("year>=2025"))

temp_silver["openf1_sessions"]=(temp_silver["openf1_sessions"].filter("year>=2025"))

temp_silver["openf1_starting_grid"]=filter_to_2025(temp_silver["openf1_starting_grid"],temp_silver["openf1_meetings"])

In [0]:
#change key to id for openf1

def rename_to_id(df):
    for col in df.columns:
        if col.endswith("_key"):
            df=df.withColumnRenamed(col,col.replace("_key","_id"))
    return df


for table_name,df in temp_silver.items():
    if table_name.startswith("openf1"):
        temp_silver[table_name]=rename_to_id(df)



In [0]:
#split session results into session type

temp_silver["openf1_race_results"] = (temp_silver["openf1_session_result"].filter(col("session_type") == "Race"))

temp_silver["openf1_qualifying_results"] = (temp_silver["openf1_session_result"].filter(col("session_type") == "Qualifying"))

temp_silver["openf1_sprint_results"] = (temp_silver["openf1_session_result"].filter(col("session_type") == "Sprint"))

In [0]:
# can finally load into silver schema!!!
for table_name,df in temp_silver.items():
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema","true")
        .saveAsTable(f"f1_warehouse.silver.{table_name}"))